# Bubble: Content-Based Music Recommender
## CM3005 Data Science Project

This notebook explores the Spotify dataset and develops multiple recommendation algorithms for the Bubble system. We'll analyze audio feature distributions, build baseline and hybrid models, and evaluate them using custom metrics.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load the Spotify dataset
df = pd.read_csv('../data/spotify_tracks.csv')
df = df.rename(columns={'artists': 'artist_name', 'track_genre': 'genre'})
if 'track_id' not in df.columns and 'Unnamed: 0' in df.columns:
    df = df.rename(columns={'Unnamed: 0': 'track_id'})
if 'genre' not in df.columns:
    df['genre'] = 'unknown'
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")

In [ ]:
# Inspect data types
print("Data types:")
print(df.dtypes)

In [ ]:
# Check for missing values
print("Missing values:")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")

In [ ]:
# Check for duplicates
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate track_ids: {df['track_id'].duplicated().sum()}")
print(f"\nFirst few rows:")
df.head()

## 2. Feature Distributions

Examining the distributions of key audio features to understand the dataset characteristics.

In [ ]:
# Select feature columns for analysis
FEATURE_COLS = ['valence', 'energy', 'acousticness', 'danceability', 
                'instrumentalness', 'speechiness', 'liveness', 'tempo', 
                'loudness']

# Summary statistics
print("Feature statistics:")
df[FEATURE_COLS].describe()

In [ ]:
# Plot distributions for key features
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribution of Key Audio Features', fontsize=16, fontweight='bold')

features_to_plot = ['valence', 'energy', 'acousticness', 'danceability']

for idx, feature in enumerate(features_to_plot):
    ax = axes[idx // 2, idx % 2]
    sns.histplot(data=df, x=feature, kde=True, ax=ax, bins=30, color='steelblue')
    ax.set_title(f'{feature.capitalize()} Distribution', fontweight='bold')
    ax.set_xlabel(feature.capitalize())
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Seaborn violin plots for better distribution visualization
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('Feature Distributions (Violin Plots)', fontsize=14, fontweight='bold')

for idx, feature in enumerate(features_to_plot):
    sns.violinplot(data=df, y=feature, ax=axes[idx], color='lightblue')
    axes[idx].set_title(feature.capitalize())

plt.tight_layout()
plt.show()

## 3. Valence vs Energy Scatter

Examining the relationship between valence (positivity) and energy, divided into quadrants with emotional interpretations.

In [ ]:
# Create valence vs energy scatter with quadrant annotations
fig, ax = plt.subplots(figsize=(14, 10))

# Use fixed 0.5 thresholds on raw valence/energy (consistent with backend).
# Spotify valence and energy are already on [0, 1], so 0.5 is the midpoint.
# These are heuristic candidate regions, NOT proven emotional labels.
valence_threshold = 0.5
energy_threshold = 0.5

# Define quadrants with different colors
q1 = df[(df['valence'] >= valence_threshold) & (df['energy'] >= energy_threshold)]
q2 = df[(df['valence'] < valence_threshold) & (df['energy'] >= energy_threshold)]
q3 = df[(df['valence'] < valence_threshold) & (df['energy'] < energy_threshold)]
q4 = df[(df['valence'] >= valence_threshold) & (df['energy'] < energy_threshold)]

# Plot each quadrant
ax.scatter(q1['valence'], q1['energy'], alpha=0.3, s=10, color='green', label='Q1: High-arousal positive')
ax.scatter(q2['valence'], q2['energy'], alpha=0.3, s=10, color='red', label='Q2: High-arousal negative')
ax.scatter(q3['valence'], q3['energy'], alpha=0.3, s=10, color='blue', label='Q3: Low-arousal negative')
ax.scatter(q4['valence'], q4['energy'], alpha=0.3, s=10, color='orange', label='Q4: Calm-positive region')

# Add quadrant lines
ax.axvline(x=valence_threshold, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax.axhline(y=energy_threshold, color='gray', linestyle='--', alpha=0.5, linewidth=1)

# Add quadrant labels
ax.text(0.75, 0.75, 'Q1: High-arousal positive', transform=ax.transAxes, 
        fontsize=11, fontweight='bold', bbox=dict(boxstyle='round', facecolor='green', alpha=0.2))
ax.text(0.05, 0.75, 'Q2: High-arousal negative', transform=ax.transAxes, 
        fontsize=11, fontweight='bold', bbox=dict(boxstyle='round', facecolor='red', alpha=0.2))
ax.text(0.05, 0.05, 'Q3: Low-arousal negative', transform=ax.transAxes, 
        fontsize=11, fontweight='bold', bbox=dict(boxstyle='round', facecolor='blue', alpha=0.2))
ax.text(0.75, 0.05, 'Q4: Calm-positive region', transform=ax.transAxes, 
        fontsize=11, fontweight='bold', bbox=dict(boxstyle='round', facecolor='orange', alpha=0.2))

ax.set_xlabel('Valence (Positivity)', fontsize=12, fontweight='bold')
ax.set_ylabel('Energy', fontsize=12, fontweight='bold')
ax.set_title('Valence vs Energy: Emotional Quadrants', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Q1 (High-arousal positive): {len(q1)} tracks")
print(f"Q2 (High-arousal negative): {len(q2)} tracks")
print(f"Q3 (Low-arousal negative): {len(q3)} tracks")
print(f"Q4 (Calm-positive region): {len(q4)} tracks")

## 4. Intimacy Score

Creating a custom intimacy score by combining normalized features. This metric captures songs that are positive yet low-energy, acoustic, and low-speech.

In [ ]:
# Normalize features for intimacy score calculation
scaler = MinMaxScaler()
normalized_features = pd.DataFrame(
    scaler.fit_transform(df[['valence', 'energy', 'acousticness', 'speechiness']]),
    columns=['valence_norm', 'energy_norm', 'acousticness_norm', 'speechiness_norm']
)

# Calculate intimacy score
# Intimate songs: positive (high valence), calm (low energy), acoustic, low speech
df['intimacy_score'] = (normalized_features['valence_norm'] * 
                         (1 - normalized_features['energy_norm']) * 
                         normalized_features['acousticness_norm'] * 
                         (1 - normalized_features['speechiness_norm']))

print(f"Intimacy score range: {df['intimacy_score'].min():.4f} to {df['intimacy_score'].max():.4f}")
print(f"Mean intimacy score: {df['intimacy_score'].mean():.4f}")
print(f"Std intimacy score: {df['intimacy_score'].std():.4f}")

In [ ]:
# Plot intimacy score distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with KDE
sns.histplot(data=df, x='intimacy_score', kde=True, ax=ax1, bins=40, color='purple')
ax1.set_title('Distribution of Intimacy Scores', fontsize=12, fontweight='bold')
ax1.set_xlabel('Intimacy Score')
ax1.set_ylabel('Frequency')

# Box plot
sns.boxplot(data=df, y='intimacy_score', ax=ax2, color='lightblue')
ax2.set_title('Intimacy Score Box Plot', fontsize=12, fontweight='bold')
ax2.set_ylabel('Intimacy Score')

plt.tight_layout()
plt.show()

In [ ]:
# Group by genre and analyze intimacy
genre_intimacy = df.groupby('genre').agg({
    'intimacy_score': ['mean', 'std', 'count']
}).round(4)

genre_intimacy.columns = ['mean_intimacy', 'std_intimacy', 'count']
genre_intimacy = genre_intimacy.sort_values('mean_intimacy', ascending=False)

print("Top 10 genres by mean intimacy score:")
print(genre_intimacy.head(10))
print("\nBottom 10 genres by mean intimacy score:")
print(genre_intimacy.tail(10))

In [ ]:
# Visualize intimacy by genre (top genres)
top_genres = df['genre'].value_counts().head(10).index
df_top_genres = df[df['genre'].isin(top_genres)]

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=df_top_genres, x='genre', y='intimacy_score', ax=ax, palette='Set2')
ax.set_title('Intimacy Score by Genre (Top 10 Genres)', fontsize=12, fontweight='bold')
ax.set_xlabel('Genre')
ax.set_ylabel('Intimacy Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Cosine Similarity Baseline

Implementing a baseline content-based recommender using cosine similarity on normalized audio features.

In [ ]:
# Prepare features for similarity computation
feature_cols_for_sim = ['valence', 'energy', 'acousticness', 'danceability', 
                        'instrumentalness', 'speechiness', 'liveness', 'tempo', 'loudness']

# Normalize features
scaler_sim = MinMaxScaler()
features_normalized = scaler_sim.fit_transform(df[feature_cols_for_sim])

# Compute cosine similarity matrix
similarity_matrix = cosine_similarity(features_normalized)
print(f"Similarity matrix shape: {similarity_matrix.shape}")
print(f"Similarity values range: {similarity_matrix.min():.4f} to {similarity_matrix.max():.4f}")

In [ ]:
# Function to get top-N recommendations using cosine similarity
def get_cosine_recommendations(track_name, artist_name, n=5):
    """Get top-N recommendations using cosine similarity baseline."""
    # Find the seed track
    mask = (df['track_name'] == track_name) & (df['artist_name'] == artist_name)
    if not mask.any():
        print(f"Track '{track_name}' by {artist_name} not found.")
        return None
    
    seed_idx = df[mask].index[0]
    
    # Get similarity scores
    similarities = similarity_matrix[seed_idx]
    
    # Get top-N (excluding the seed itself)
    top_indices = np.argsort(-similarities)[1:n+1]
    
    results = df.iloc[top_indices][['track_name', 'artist_name', 'genre']].copy()
    results['similarity_score'] = similarities[top_indices]
    
    return results, seed_idx

# Test with example seeds
seed_songs = [
    df.iloc[0],
    df.iloc[len(df)//3],
    df.iloc[2*len(df)//3]
]

print("\n" + "="*80)
print("COSINE SIMILARITY BASELINE RECOMMENDATIONS")
print("="*80)

for i, seed in enumerate(seed_songs, 1):
    print(f"\nSeed {i}: '{seed['track_name']}' by {seed['artist_name']} (Genre: {seed['genre']})")
    print("-" * 80)
    
    recs, seed_idx = get_cosine_recommendations(seed['track_name'], seed['artist_name'], n=5)
    
    if recs is not None:
        for idx, (_, row) in enumerate(recs.iterrows(), 1):
            print(f"  {idx}. '{row['track_name']}' by {row['artist_name']} "
                  f"(Genre: {row['genre']}, Score: {row['similarity_score']:.4f})")

## 6. KNN and Hybrid Recommenders

Implementing KNN-based recommendations and a hybrid approach combining content similarity with intimacy scores.

In [ ]:
# KNN Recommender using cosine similarity
knn = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='brute')
knn.fit(features_normalized)

def get_knn_recommendations(track_name, artist_name, n=5):
    """Get top-N recommendations using KNN."""
    mask = (df['track_name'] == track_name) & (df['artist_name'] == artist_name)
    if not mask.any():
        print(f"Track '{track_name}' by {artist_name} not found.")
        return None
    
    seed_idx = df[mask].index[0]
    seed_vector = features_normalized[seed_idx].reshape(1, -1)
    
    # Query KNN
    distances, indices = knn.kneighbors(seed_vector)
    
    # Convert distances to similarity (1 / (1 + distance))
    similarities = 1 / (1 + distances[0][1:n+1])
    indices = indices[0][1:n+1]
    
    results = df.iloc[indices][['track_name', 'artist_name', 'genre']].copy()
    results['similarity_score'] = similarities
    
    return results, seed_idx

print("KNN Recommender initialized and ready.")

In [ ]:
# Hybrid Recommender: weighted combination of content similarity and intimacy
def get_hybrid_recommendations(track_name, artist_name, n=5, alpha=0.7):
    """Get top-N recommendations using hybrid approach.
    
    Args:
        track_name: Name of seed track
        artist_name: Name of seed artist
        n: Number of recommendations
        alpha: Weight for content similarity (1-alpha for intimacy)
    """
    mask = (df['track_name'] == track_name) & (df['artist_name'] == artist_name)
    if not mask.any():
        print(f"Track '{track_name}' by {artist_name} not found.")
        return None
    
    seed_idx = df[mask].index[0]
    
    # Content similarity
    content_sim = similarity_matrix[seed_idx]
    
    # Normalize intimacy score to [0, 1]
    intimacy_min = df['intimacy_score'].min()
    intimacy_max = df['intimacy_score'].max()
    intimacy_norm = (df['intimacy_score'] - intimacy_min) / (intimacy_max - intimacy_min)
    
    # Hybrid score
    hybrid_score = alpha * content_sim + (1 - alpha) * intimacy_norm.values
    
    # Get top-N (excluding seed)
    top_indices = np.argsort(-hybrid_score)[1:n+1]
    
    results = df.iloc[top_indices][['track_name', 'artist_name', 'genre', 'intimacy_score']].copy()
    results['hybrid_score'] = hybrid_score[top_indices]
    
    return results, seed_idx

print("Hybrid Recommender initialized and ready.")

In [ ]:
# Compare the three approaches for the same seeds
print("\n" + "="*100)
print("COMPARISON: COSINE vs KNN vs HYBRID")
print("="*100)

for i, seed in enumerate(seed_songs[:2], 1):  # Use first 2 seeds for brevity
    print(f"\n{'='*100}")
    print(f"SEED {i}: '{seed['track_name']}' by {seed['artist_name']}")
    print(f"{'='*100}")
    
    # Cosine
    print("\nCOSINE SIMILARITY:")
    print("-" * 100)
    cos_recs, _ = get_cosine_recommendations(seed['track_name'], seed['artist_name'], n=5)
    for idx, (_, row) in enumerate(cos_recs.iterrows(), 1):
        print(f"  {idx}. '{row['track_name']}' by {row['artist_name']} (Score: {row['similarity_score']:.4f})")
    
    # KNN
    print("\nKNN (k=5):")
    print("-" * 100)
    knn_recs, _ = get_knn_recommendations(seed['track_name'], seed['artist_name'], n=5)
    for idx, (_, row) in enumerate(knn_recs.iterrows(), 1):
        print(f"  {idx}. '{row['track_name']}' by {row['artist_name']} (Score: {row['similarity_score']:.4f})")
    
    # Hybrid
    print("\nHYBRID (alpha=0.7):")
    print("-" * 100)
    hyb_recs, _ = get_hybrid_recommendations(seed['track_name'], seed['artist_name'], n=5, alpha=0.7)
    for idx, (_, row) in enumerate(hyb_recs.iterrows(), 1):
        print(f"  {idx}. '{row['track_name']}' by {row['artist_name']} (Score: {row['hybrid_score']:.4f})")

## 7. Evaluation Metrics

Implementing custom evaluation metrics for the recommender systems: precision@K and intra-list diversity.

In [ ]:
# Evaluation metric: Precision@K (using genre as proxy)
def precision_at_k(recommendations, seed_genre, k=5):
    """Calculate precision@K using genre matching as proxy.
    
    Genre matching is an imperfect but practical proxy for relevance.
    """
    recs_k = recommendations.head(k)
    genre_matches = (recs_k['genre'] == seed_genre).sum()
    precision = genre_matches / k
    return precision

# Evaluation metric: Intra-list diversity
def intra_list_diversity(recommendations, indices_in_full_set):
    """Calculate average pairwise cosine distance within recommendation list.
    
    Higher diversity means more different recommendations.
    """
    rec_indices = [df.index.get_loc(i) for i in indices_in_full_set]
    rec_features = features_normalized[rec_indices]
    
    # Compute pairwise cosine distances
    pairwise_sim = cosine_similarity(rec_features)
    
    # Average distance (1 - similarity)
    n = len(rec_indices)
    distances = []
    for i in range(n):
        for j in range(i+1, n):
            distances.append(1 - pairwise_sim[i, j])
    
    return np.mean(distances) if distances else 0.0

print("Evaluation metrics defined.")

In [ ]:
# Evaluate all three recommenders on 5 different seeds
np.random.seed(42)
seed_indices = np.random.choice(len(df), size=5, replace=False)

evaluation_results = []

for i, seed_idx in enumerate(seed_indices, 1):
    seed = df.iloc[seed_idx]
    track_name = seed['track_name']
    artist_name = seed['artist_name']
    seed_genre = seed['genre']
    
    print(f"\nEvaluating seed {i}: '{track_name}' by {artist_name}")
    
    # Get recommendations from all three methods
    cos_recs, _ = get_cosine_recommendations(track_name, artist_name, n=5)
    knn_recs, _ = get_knn_recommendations(track_name, artist_name, n=5)
    hyb_recs, _ = get_hybrid_recommendations(track_name, artist_name, n=5, alpha=0.7)
    
    # Compute metrics
    if cos_recs is not None:
        cos_precision = precision_at_k(cos_recs, seed_genre, k=5)
        cos_diversity = intra_list_diversity(cos_recs, cos_recs.index)
        
        knn_precision = precision_at_k(knn_recs, seed_genre, k=5)
        knn_diversity = intra_list_diversity(knn_recs, knn_recs.index)
        
        hyb_precision = precision_at_k(hyb_recs, seed_genre, k=5)
        hyb_diversity = intra_list_diversity(hyb_recs, hyb_recs.index)
        
        evaluation_results.append({
            'Seed': i,
            'Track': track_name[:30],
            'Genre': seed_genre,
            'Cosine_Precision': cos_precision,
            'Cosine_Diversity': cos_diversity,
            'KNN_Precision': knn_precision,
            'KNN_Diversity': knn_diversity,
            'Hybrid_Precision': hyb_precision,
            'Hybrid_Diversity': hyb_diversity
        })

In [ ]:
# Display evaluation results
eval_df = pd.DataFrame(evaluation_results)

print("\n" + "="*120)
print("EVALUATION RESULTS: PRECISION@5 AND INTRA-LIST DIVERSITY")
print("="*120)
print("\n" + eval_df.to_string(index=False))

print("\n" + "-"*120)
print("SUMMARY STATISTICS:")
print("-"*120)

summary = pd.DataFrame({
    'Metric': ['Precision@5', 'Precision@5', 'Precision@5', 'Intra-List Diversity', 'Intra-List Diversity', 'Intra-List Diversity'],
    'Method': ['Cosine', 'KNN', 'Hybrid', 'Cosine', 'KNN', 'Hybrid'],
    'Mean': [
        eval_df['Cosine_Precision'].mean(),
        eval_df['KNN_Precision'].mean(),
        eval_df['Hybrid_Precision'].mean(),
        eval_df['Cosine_Diversity'].mean(),
        eval_df['KNN_Diversity'].mean(),
        eval_df['Hybrid_Diversity'].mean()
    ],
    'Std': [
        eval_df['Cosine_Precision'].std(),
        eval_df['KNN_Precision'].std(),
        eval_df['Hybrid_Precision'].std(),
        eval_df['Cosine_Diversity'].std(),
        eval_df['KNN_Diversity'].std(),
        eval_df['Hybrid_Diversity'].std()
    ]
})

print("\n" + summary.to_string(index=False))

In [ ]:
# Visualize evaluation results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision comparison
precision_data = pd.DataFrame({
    'Cosine': eval_df['Cosine_Precision'],
    'KNN': eval_df['KNN_Precision'],
    'Hybrid': eval_df['Hybrid_Precision']
})

precision_data.plot(kind='bar', ax=axes[0], color=['steelblue', 'coral', 'mediumseagreen'])
axes[0].set_title('Precision@5 by Recommender', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Precision')
axes[0].set_xlabel('Seed Track')
axes[0].set_ylim([0, 1])
axes[0].legend(title='Method')
axes[0].grid(axis='y', alpha=0.3)

# Diversity comparison
diversity_data = pd.DataFrame({
    'Cosine': eval_df['Cosine_Diversity'],
    'KNN': eval_df['KNN_Diversity'],
    'Hybrid': eval_df['Hybrid_Diversity']
})

diversity_data.plot(kind='bar', ax=axes[1], color=['steelblue', 'coral', 'mediumseagreen'])
axes[1].set_title('Intra-List Diversity by Recommender', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Average Pairwise Distance')
axes[1].set_xlabel('Seed Track')
axes[1].legend(title='Method')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Discussion

Key findings and insights from the analysis, along with planned improvements for the Bubble recommender system.

### Findings on Emotional Quadrants

The valence-energy scatter plot reveals distinct clusters of music across emotional dimensions:

- **Q1 (High-arousal positive)**: Features high-energy, positive tracks. These dominate in pop and electronic genres.
- **Q2 (High-arousal negative)**: Low valence but high energy; common in rock, metal, and hip-hop. Contains aggressive, intense tracks.
- **Q3 (Low-arousal negative)**: Both low valence and low energy. Typical in folk, blues, and slow ballads.
- **Q4 (Tender/Warm)**: High valence but low energy. Contains intimate, acoustic, and singer-songwriter material.

The Q4 cluster appears most relevant to the "intimacy score" concept, with its combination of positivity and calmness ideal for intimate listening contexts.

### Evaluation Metrics: Precision and Diversity

**Precision@K (using genre as proxy):**
- Genre matching is an imperfect but practical proxy for relevance in music recommendation.
- It cannot capture intra-genre variation or cross-genre similarity (e.g., acoustic rock vs. folk).
- Future work should incorporate user ratings or implicit feedback for ground truth.

**Intra-List Diversity:**
- Measures the average pairwise cosine distance within the top-5 recommendations.
- Higher diversity prevents "echo chamber" recommendations and exposes users to varied music.
- The hybrid recommender balances similarity and diversity through the intimacy score component.

### Intimacy Score Behavior

The custom intimacy score combines:
- **Valence**: Emotional positivity
- **Energy**: Activity level (inverse)
- **Acousticness**: Organic instrumentation
- **Speechiness**: Minimal spoken content (inverse)

This metric captures the "tender, warm" feeling of Q4 tracks. Top-scoring genres (acoustic, soul, singer-songwriter) validate the approach. However, the metric is domain-specific and should be refined with user feedback.

### Recommender Comparison

**Cosine Similarity (Baseline):**
- Simple, interpretable, fast for small datasets.
- Computes pairwise similarity on normalized features.
- Performs well for discovering acoustically similar tracks.

**KNN:**
- Finds k-nearest neighbors in the feature space.
- Nearly identical to cosine similarity for this application.
- Better scalability with optimized libraries (KD-trees, LSH).

**Hybrid:**
- Combines content similarity (70%) with intimacy context (30%).
- Boosts diverse, emotionally-aligned recommendations.
- Alpha parameter allows fine-tuning the balance.

### Planned Improvements

1. **FAISS Integration:**
   - Use Facebook AI Similarity Search for million-scale datasets.
   - Enables approximate nearest neighbor search with sub-millisecond latency.

2. **User Feedback Loop:**
   - Collect implicit signals (skip, play duration, favorite-toggle).
   - Retrain or adapt model weights based on user preferences.

3. **Collaborative Filtering:**
   - Combine content-based approach with user-user or item-item similarity.
   - Addresses cold-start with content, discovers serendipitous recommendations via CF.

4. **Context Awareness:**
   - Incorporate time-of-day, activity type, and seasonal factors.
   - Example: Suggest high-energy tracks for morning workouts, low-energy for evening relaxation.

5. **Fine-tuning Features:**
   - Engineer genre embeddings to capture semantic relationships.
   - Weight features based on user preferences and listening history.

---

# Iteration 2: Weighted Similarity, Destination Mode, MMR, and Batch Evaluation

This section implements and evaluates improvements to the content-based recommender.

## Hypotheses

1. **Equal feature weighting is suboptimal.** All nine Spotify audio features are currently weighted equally, despite uncertain validity of danceability for relationship-context relevance. We hypothesise that emphasising valence and energy (the two features most tied to Russell's circumplex) while reducing danceability will improve genre-match precision.

2. **Cosine and KNN return low-diversity lists.** We add MMR (Maximum Marginal Relevance) reranking to trade a small amount of relevance for increased intra-list diversity.

3. **Hard Q4 candidate restriction is counterintuitive for non-Q4 seeds.** We replace the emotional_filter hard cutoff with a soft destination-mode scoring bonus that rewards calm-positive tracks continuously, without excluding candidates from other quadrants.

4. **Single-list coverage was misleadingly labelled 'catalogue coverage'.** True catalogue coverage must be computed across many seeds. We add a batch evaluation function that samples 100+ seeds and aggregates metrics.

## Important caveats

- **Genre matching is only an offline proxy** for relevance. It cannot capture intra-genre variation or cross-genre similarity.
- **User ratings are needed** for final relevance validation. No user study has been conducted. We report only measured offline results.
- **Russell's Circumplex Model and Spotify valence/energy are a heuristic framework**, not ground-truth emotional labels. Q4 is a 'calm-positive destination region', not a proven intimacy label.
- **Feature weights are hand-designed experimental profiles**, not learned or personalised weights.


## Iteration 2: Setup

Import the iteration-2 backend modules and load the recommender with the full dataset.

In [ ]:
import sys
sys.path.insert(0, '..')

from app.recommender import (
    BubbleRecommender, FEATURE_COLS, FEATURE_WEIGHT_PROFILES,
    assign_quadrant, validate_weights,
)
from app.evaluation import (
    evaluate_batch, compare_configurations,
    precision_at_k, intra_list_diversity, catalogue_coverage,
)

rec = BubbleRecommender()
rec.load('../data/spotify_tracks.csv')
print(f'Loaded {rec.track_count} tracks')
print(f'Feature weight profiles: {list(FEATURE_WEIGHT_PROFILES.keys())}')


## Iteration 2: Feature Weight Profiles

Three named profiles are available:

| Profile | Description |
|---|---|
| `equal` | All weights 1.0 — iteration-1 baseline |
| `no_danceability` | Danceability weight 0.0, all others 1.0 — ablation |
| `affect_emphasis` | Valence 2.0, energy 2.0, acousticness 1.5, speechiness 1.5, danceability 0.5, instrumentalness 0.5, tempo 0.5, loudness 0.5, liveness 0.3 |

Weighted cosine is implemented by multiplying both the seed and candidate vectors by sqrt(weight) before computing cosine similarity.

In [ ]:
# Display the weight profiles
for name, weights in FEATURE_WEIGHT_PROFILES.items():
    print(f'\n{name}:')
    for feat, w in weights.items():
        print(f'  {feat:20s}: {w:.1f}')


## Iteration 2: Batch Evaluation — Four Configurations

We evaluate four configurations across 100 seeds (random_state=42):

1. **iteration1_baseline**: equal weights, cosine, no destination, MMR off
2. **ablation_no_danceability**: no_danceability profile, cosine, MMR off
3. **hybrid_affect_emphasis**: affect_emphasis, alpha=0.7, MMR off
4. **hybrid_affect_emphasis_mmr**: affect_emphasis, alpha=0.7, MMR on (lambda=0.75)


In [ ]:
# Run batch comparison across 100 seeds
import os

results_df = compare_configurations(
    rec,
    n_seeds=100,
    k=10,
    random_state=42,
    output_csv='iteration2_batch_results.csv',
)

print('\nBatch evaluation results (100 seeds):')
print(results_df.to_string(index=False))


## Iteration 2: Results Visualisation

Comparing mean Precision@K and intra-list diversity across the four configurations.

In [ ]:
# Chart: Mean Precision@K and Intra-list Diversity across configurations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision@K
axes[0].barh(results_df['config_name'], results_df['precision_at_k_mean'],
             xerr=results_df['precision_at_k_std'], color='steelblue', alpha=0.8)
axes[0].set_xlabel('Mean Precision@K (genre-match proxy)')
axes[0].set_title('Precision@K by Configuration', fontweight='bold')
axes[0].set_xlim(0, 1)

# Intra-list diversity
axes[1].barh(results_df['config_name'], results_df['intra_list_diversity_mean'],
             xerr=results_df['intra_list_diversity_std'], color='coral', alpha=0.8)
axes[1].set_xlabel('Mean Intra-list Diversity')
axes[1].set_title('Intra-list Diversity by Configuration', fontweight='bold')
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.savefig('iteration2_comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print('Chart saved to iteration2_comparison_chart.png')


In [ ]:
# Catalogue coverage comparison
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(results_df['config_name'], results_df['catalogue_coverage'],
       color='mediumseagreen', alpha=0.8)
ax.set_ylabel('Catalogue Coverage')
ax.set_title('Catalogue Coverage by Configuration (100 seeds)', fontweight='bold')
ax.set_xticklabels(results_df['config_name'], rotation=30, ha='right')
plt.tight_layout()
plt.show()


## Iteration 2: What Changed from Iteration 1

| Aspect | Iteration 1 | Iteration 2 |
|---|---|---|
| Feature weights | All equal (1.0) | Three named profiles (equal, no_danceability, affect_emphasis) |
| Quadrant thresholds | Backend: 0.5, Notebook: medians | Fixed 0.5 everywhere |
| Q4 filter | Hard candidate restriction | Soft destination_mode scoring bonus (calm_positive) |
| Diversity | None | MMR reranking (configurable lambda) |
| Coverage | Single-list, mislabelled | Catalogue coverage across many seeds |
| Intra-list diversity | 4 features only | All 9 weighted features |
| Batch evaluation | 5 seeds, 3 methods | 100 seeds, 4 configurations with mean/std |
| Affect labels | 'Tender/Warm', 'Happy/Excited' | 'Calm-positive destination region', heuristic candidate regions |

## Iteration 2: Discussion — Did Each Change Improve, Worsen, or Trade Off?

The batch results table and charts above show the measured offline metrics. Key observations:

1. **no_danceability ablation**: Removing danceability from the similarity calculation tests whether it contributes to relationship-context relevance. If Precision@K increases, danceability was adding noise; if it decreases, danceability was contributing useful signal.

2. **affect_emphasis hybrid**: Emphasising valence and energy while de-emphasising danceability and liveness tests the hypothesis that affect-related features are more important for relationship-context recommendations. The hybrid method also adds the intimacy score component.

3. **MMR reranking**: By design, MMR trades a small amount of relevance for increased diversity. We expect Precision@K to decrease slightly and intra-list diversity to increase. The lambda parameter (default 0.75) controls this trade-off.

4. **Catalogue coverage**: Shows how much of the catalogue each configuration surfaces across 100 seeds. Higher coverage means more diverse overall recommendations.

**None of these results should be interpreted as proof of improved subjective relevance.** Genre matching is an imperfect offline proxy. User ratings from a consent-based participant study are needed to validate whether these changes actually improve the user experience.


## Iteration 2: Known Limitations and Next Step

**Limitations:**
- Genre matching is a crude proxy — two tracks in the same genre can feel very different, and cross-genre recommendations can be highly relevant.
- Feature weights are hand-designed, not learned from user preference data.
- Russell's circumplex is a heuristic framework; valence and energy do not perfectly capture emotional experience.
- No user study has been conducted. All claims are based on offline metrics only.

**Next step: consent-based participant relevance study.**
Recruit participants who consent to providing relevance ratings for recommended tracks. Use these ratings (not genre matching) as the ground-truth signal to evaluate whether iteration-2 changes improve perceived recommendation quality.
